In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
from scipy import stats

In [2]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

,date_time,user_id,is_mobile,is_package,channel,cnt,trip_type,is_family_trip,is_multi_room,total_guests,...,distance_clean,distance_behavior_group,user_booking_rate,avg_session_intensity,avg_stay_duration,avg_advance_booking_days,mobile_usage_rate,destination_popularity,hotel_cluster,is_booking
0,2014-04-27 19:43:09,14,0,0,9,1,solo,0,0,1,...,2265.9330,far,0.0,1.0,2.0,54.0,0.0,3544,88,0
1,2014-10-22 08:51:12,38,0,0,2,1,group,0,0,2,...,480.7833,medium,0.0,1.0,4.0,123.0,0.0,47,33,0
2,2014-08-05 16:21:41,40,0,0,0,1,solo,0,0,1,...,2576.2959,far,0.0,1.0,1.0,1.0,0.0,100,76,0
3,2014-09-18 22:30:51,156,0,0,5,1,family,1,0,4,...,1128.2299,far,0.0,1.0,1.0,14.0,0.0,214,13,0
4,2014-09-18 22:24:52,156,0,0,5,1,family,1,0,4,...,1127.9475,far,0.0,1.0,1.0,14.0,0.0,214,16,0


In [3]:
features = [
    "is_mobile",
    "cnt",
    "advance_booking_days",
    "is_distance_unknown",
    "trip_type",
    "channel"
]

df_model = df[features + ["is_booking"]].copy()

df_model = pd.get_dummies(df_model, drop_first=True)

df_model = df_model.dropna()

X = df_model.drop("is_booking", axis=1)
y = df_model["is_booking"]

X = X.astype(float)

X = sm.add_constant(X)

model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.253368
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:             is_booking   No. Observations:                99471
Model:                          Logit   Df Residuals:                    99463
Method:                           MLE   Df Model:                            7
Date:                Wed, 22 Apr 2026   Pseudo R-squ.:                 0.09378
Time:                        01:40:59   Log-Likelihood:                -25203.
converged:                       True   LL-Null:                       -27811.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.7078      0.096      7.356      0.000       0.519       0.89

In [4]:
conn.close()

## Logistic Regression Insights

A logistic regression model was used to control for key behavioral and contextual variables.

The results confirm that mobile usage has a significant negative impact on booking probability:

- Mobile coefficient: -0.277 (p < 0.001)
- Odds ratio: ~0.76

This indicates that mobile users are approximately 24% less likely to complete a booking compared to desktop users, even after controlling for session intensity, booking window, distance availability, and trip characteristics.

These findings are consistent with the observational A/B test, which showed a ~27% lower conversion rate for mobile users.

## Additional Insights

- Higher session intensity (`cnt`) is associated with lower conversion, suggesting exploratory behavior.
- Longer booking windows reduce conversion likelihood.
- Missing distance information negatively impacts booking probability.
- Solo travelers show higher conversion rates compared to groups.

## Conclusion

The negative effect of mobile on conversion appears to be robust and not explained by observable behavioral variables, suggesting structural differences in user experience or context across devices.